<a href="https://colab.research.google.com/github/Ramy99999999/Flyrank-machine-learning-internship/blob/main/work/notebooks/w04_baseline_score.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Ramy99999999/Flyrank-machine-learning-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [9]:
# Week-4 setup: recreate the Week-3 data contract

%pip install -q duckdb huggingface_hub pandas scikit-learn

import os
import duckdb
import pandas as pd
import numpy as np

from google.colab import userdata

# Get Hugging Face token
HF_TOKEN = userdata.get("HF_TOKEN")

# Connect DuckDB
con = duckdb.connect()
con.execute("SET enable_progress_bar = false")
con.execute("SET VARIABLE hf_token = ?", [HF_TOKEN])

con.execute("""
    CREATE OR REPLACE SECRET hf (
        TYPE huggingface,
        TOKEN getvariable('hf_token')
    )
""")

# Warehouse locations
REL = "hf://datasets/FlyRank/internship-warehouse"
FACT = f"{REL}/fact_content_daily_performance"
DIM_CONTENT = f"{REL}/dim_content.parquet"
DIM_CLIENTS = f"{REL}/dim_clients.parquet"

# Time windows
FEB = f"read_parquet('{FACT}/month=2026-02/*.parquet')"
MAR = f"read_parquet('{FACT}/month=2026-03/*.parquet')"

# Build February universe
universe_feb = con.execute(f"""
SELECT
    f.client_hash_id,
    f.content_hash_id
FROM {FEB} AS f
JOIN read_parquet('{DIM_CONTENT}') AS d
    ON f.client_hash_id = d.client_hash_id
    AND f.content_hash_id = d.content_hash_id
GROUP BY
    f.client_hash_id,
    f.content_hash_id,
    d.is_published,
    d.content_created_date
HAVING
    SUM(f.gsc_impressions) >= 100
    AND SUM(f.gsc_clicks) >= 3
    AND d.is_published IS TRUE
    AND d.content_created_date <= DATE '2026-02-28'
""").df()

# Build February features
features_feb = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,
    SUM(gsc_impressions) AS gsc_impressions_feb,
    SUM(gsc_clicks) AS gsc_clicks_feb,

    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN gsc_sum_position
            ELSE 0
        END
    ) / NULLIF(
        SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN gsc_impressions
                ELSE 0
            END
        ),
        0
    ) AS avg_position_feb,

    SUM(sessions_organic) AS sessions_organic_feb,
    SUM(scroll_events) AS scroll_events_feb

FROM {FEB}
GROUP BY
    client_hash_id,
    content_hash_id
""").df()

# Build March outcome label
labels_mar = con.execute(f"""
SELECT
    client_hash_id,
    content_hash_id,

    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS measured_days_mar,

    SUM(gsc_clicks) FILTER (
        WHERE gsc_data_available IS TRUE
    ) AS clicks_mar,

    CASE
        WHEN SUM(gsc_clicks) FILTER (
            WHERE gsc_data_available IS TRUE
        ) = 0
        THEN 1
        ELSE 0
    END AS went_dark

FROM {MAR}
GROUP BY
    client_hash_id,
    content_hash_id

HAVING
    COUNT(*) FILTER (
        WHERE gsc_data_available IS TRUE
    ) > 0
""").df()

# Final Week-3 analysis frame
final_frame = (
    features_feb
    .merge(
        universe_feb,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
    .merge(
        labels_mar,
        on=["client_hash_id", "content_hash_id"],
        how="inner"
    )
)

print("final_frame shape:", final_frame.shape)
print("went_dark rate:", final_frame["went_dark"].mean())

final_frame shape: (29353, 10)
went_dark rate: 0.03948489081184206


## 1. My rule and its reason codes

**Rule:** Prioritize published content with lower February search visibility. The score gives higher priority to pages with fewer February GSC impressions because the signal audit showed a strong, consistent relationship between lower impression volume and a higher observed March `went_dark` rate.

**Reason code:** `low_search_visibility_risk`

**Action:** `review_low_visibility`

The CTR-gap signal was also tested against March `went_dark`, but its observed relationship did not support the intended direction. It is therefore not used in the baseline score.

The rule uses only February information available at the decision point. March `went_dark` is used only for signal auditing and evaluation, never as a scoring input.



In [10]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import numpy as np
import pandas as pd

# Work only with February features plus the already-defined March outcome
audit = final_frame.copy()

# CTR from February observations
audit["ctr_feb"] = (
    audit["gsc_clicks_feb"] /
    audit["gsc_impressions_feb"].replace(0, np.nan)
).fillna(0)

# Position tiers used to compare CTR with pages at similar positions
audit["position_tier"] = pd.cut(
    audit["avg_position_feb"],
    bins=[0, 3, 10, 20, 100, np.inf],
    labels=["1-3", "4-10", "11-20", "21-100", "100+"],
    include_lowest=True
)

# Expected CTR = median CTR among pages in the same position tier
tier_ctr = audit.groupby("position_tier", observed=True)["ctr_feb"].transform("median")

audit["ctr_gap"] = audit["ctr_feb"] - tier_ctr

print("SIGNAL CHECK 1 — VOLUME")
volume_bins = [-np.inf, 499, 1999, 9999, np.inf]
volume_labels = ["<500", "500-1,999", "2,000-9,999", "10,000+"]

audit["volume_bucket"] = pd.cut(
    audit["gsc_impressions_feb"],
    bins=volume_bins,
    labels=volume_labels
)

volume_table = (
    audit.groupby("volume_bucket", observed=True)
    .agg(
        n=("went_dark", "size"),
        went_dark_rate=("went_dark", "mean")
    )
    .reset_index()
)

print(volume_table.to_string(index=False))

print("\nSIGNAL CHECK 2 — CTR GAP VS POSITION")
ctr_bins = [-np.inf, -0.02, -0.005, 0.005, 0.02, np.inf]
ctr_labels = [
    "far below",
    "below",
    "near expected",
    "above",
    "far above"
]

audit["ctr_gap_bucket"] = pd.cut(
    audit["ctr_gap"],
    bins=ctr_bins,
    labels=ctr_labels
)

ctr_table = (
    audit.groupby("ctr_gap_bucket", observed=True)
    .agg(
        n=("went_dark", "size"),
        went_dark_rate=("went_dark", "mean")
    )
    .reset_index()
)

print(ctr_table.to_string(index=False))

SIGNAL CHECK 1 — VOLUME
volume_bucket     n  went_dark_rate
         <500  1769        0.152629
    500-1,999  9386        0.067228
  2,000-9,999 14933        0.016741
      10,000+  3265        0.002450

SIGNAL CHECK 2 — CTR GAP VS POSITION
ctr_gap_bucket     n  went_dark_rate
 near expected 25277        0.036713
         above  3648        0.055373
     far above   428        0.067757


### Signal verdicts

**Volume — CONFIRMED:** The measured `went_dark` rate decreases consistently as February impression volume increases: 15.26% for fewer than 500 impressions, 6.72% for 500–1,999, 1.67% for 2,000–9,999, and 0.25% for 10,000+. This supports volume as a useful directional signal for prioritization.

**CTR gap vs position — OPPOSITE:** In the observed buckets, `went_dark` is actually higher for pages above the expected CTR benchmark than for pages near the expected benchmark (5.54% and 6.78% versus 3.67%). No below-expected buckets appear in the output. Therefore, the observed relationship does not support using a negative CTR gap as evidence that a page is more likely to go dark. I will not rely on this signal in the baseline score.

The signal checks are observational: March `went_dark` is used only to test the February signals and is not used as a scoring input.



## 2. Build the ranked queue (writes the CSV)

The baseline uses the confirmed February volume signal. Pages with lower February GSC impressions receive a higher priority because the signal audit showed a strong, consistent relationship between lower search visibility and a higher observed March `went_dark` rate.

The score is:

`score = 1 / log1p(gsc_impressions_feb)`

The logarithm reduces the effect of very large impression counts, while the inverse makes lower-volume pages receive higher scores.

**Reason code:** `low_search_visibility_risk`

**Action:** `review_low_visibility`

The CTR-gap signal was audited but excluded from the score because its observed relationship was opposite to the intended direction.

Only February information is used to construct the score. March `went_dark` is retained only for evaluation and audit.



In [11]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Build the transparent baseline score.
# Only February information is used.

# Build the transparent baseline score.
# Only February information is used.

baseline = audit[
    [
        "client_hash_id",
        "content_hash_id",
        "gsc_impressions_feb",
        "gsc_clicks_feb",
        "avg_position_feb",
        "ctr_feb",
        "ctr_gap",
        "went_dark"
    ]
].copy()

# Confirmed signal:
# lower February search volume was associated with higher went_dark risk.
# Use the inverse of log-transformed impressions so lower-volume pages rank higher.
baseline["score"] = 1 / np.log1p(
    baseline["gsc_impressions_feb"]
)

baseline["reason_code"] = "low_search_visibility_risk"
baseline["action"] = "review_low_visibility"

# Rank highest score first.
baseline = baseline.sort_values(
    ["score", "gsc_impressions_feb"],
    ascending=[False, True]
).reset_index(drop=True)

baseline["rank"] = np.arange(1, len(baseline) + 1)

# Evaluation only — March went_dark is NOT used to construct the score.
def precision_at_k(df, k):
    return df.head(k)["went_dark"].mean()

print(f"Base rate: {baseline['went_dark'].mean():.4f}")
print(f"Precision@10: {precision_at_k(baseline, 10):.4f}")
print(f"Precision@20: {precision_at_k(baseline, 20):.4f}")
print(f"Precision@50: {precision_at_k(baseline, 50):.4f}")

# Required ranked queue.
output_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "reason_code",
    "action",
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "avg_position_feb",
    "ctr_feb",
    "ctr_gap"
]

queue = baseline[output_cols].copy()

import os
os.makedirs("work/outputs", exist_ok=True)

queue.to_csv(
    "work/outputs/baseline_action_score.csv",
    index=False
)

print("\nWrote:")
print("work/outputs/baseline_action_score.csv")

queue.head(20)

Base rate: 0.0395
Precision@10: 0.2000
Precision@20: 0.2000
Precision@50: 0.1800

Wrote:
work/outputs/baseline_action_score.csv


,rank,client_hash_id,content_hash_id,score,reason_code,action,gsc_impressions_feb,gsc_clicks_feb,avg_position_feb,ctr_feb,ctr_gap
0,1,client_65de48885f4ef01b,content_3d3f76287d8a0cb4,0.216679,low_search_visibility_risk,review_low_visibility,100.0,3.0,8.080000,0.030000,0.026203
1,2,client_23a62021009f63c4,content_666722e0d2f4dec0,0.216679,low_search_visibility_risk,review_low_visibility,100.0,3.0,3.840000,0.030000,0.026203
2,3,client_23a62021009f63c4,content_df0749df761a54e6,0.216679,low_search_visibility_risk,review_low_visibility,100.0,4.0,24.420000,0.040000,0.037912
3,4,client_3ffa76342f366962,content_071584d02651c65e,0.216217,low_search_visibility_risk,review_low_visibility,101.0,7.0,5.980198,0.069307,0.065509
4,5,client_20259bd6705d81d4,content_ce070943bbb0ffa5,0.216217,low_search_visibility_risk,review_low_visibility,101.0,4.0,5.871287,0.039604,0.035806
5,6,client_3ffa76342f366962,content_8cf83f905a2c7f95,0.215762,low_search_visibility_risk,review_low_visibility,102.0,4.0,6.872549,0.039216,0.035418
6,7,client_3ffa76342f366962,content_7b17975c58745266,0.215762,low_search_visibility_risk,review_low_visibility,102.0,5.0,4.450980,0.049020,0.045222
7,8,client_2094c6eb080311d5,content_696689d65af472ce,0.215762,low_search_visibility_risk,review_low_visibility,102.0,3.0,4.666667,0.029412,0.025614
8,9,client_2094c6eb080311d5,content_e4163214c04a65d8,0.215762,low_search_visibility_risk,review_low_visibility,102.0,6.0,2.196078,0.058824,0.054995
9,10,client_a80fca3f171ed1de,content_47b9e3d74d302801,0.215762,low_search_visibility_risk,review_low_visibility,102.0,3.0,13.852941,0.029412,0.025326


## 3. Top-20 review

The baseline ranks pages primarily by low February search visibility. The top 20 items are therefore pages with the lowest February GSC impression volumes. They are prioritized because the signal audit showed that lower-volume pages had a substantially higher observed `went_dark` rate.

For each reviewed item, the baseline recommendation could still be wrong if the low impression volume is caused by factors unrelated to content quality or refresh need, such as limited search demand, a narrow topic, or incomplete measurement.

1. **Rank 1:** Review for low search visibility. It has 100 February impressions and is prioritized because it falls at the lowest observed volume level. The recommendation could be wrong if the low volume reflects limited demand rather than a content problem.
2. **Rank 2:** Review for low search visibility. It has 100 February impressions and is prioritized by the same confirmed volume-risk signal. It could be wrong if the page naturally has limited search demand.
3. **Rank 3:** Review for low search visibility. It has 100 February impressions and is prioritized because low-volume pages showed higher `went_dark` risk. It could be wrong if its low volume is expected for the topic.
4. **Rank 4:** Review for low search visibility. It has 101 February impressions and is near the lowest-volume boundary. It could be wrong if the page's limited visibility is not caused by a refreshable content issue.
5. **Rank 5:** Review for low search visibility. It has 101 February impressions and is prioritized by the volume signal. It could be wrong if low impressions reflect low overall search demand.
6. **Rank 6:** Review for low search visibility. It has 102 February impressions and remains in the highest-priority low-volume group. It could be wrong if the page is performing appropriately for its niche.
7. **Rank 7:** Review for low search visibility. It has 102 February impressions and receives a high score because of its low volume. It could be wrong if measurement or search demand explains the low visibility.
8. **Rank 8:** Review for low search visibility. It has 102 February impressions and is prioritized by the same observed volume relationship. It could be wrong if the page does not have a meaningful refresh opportunity.
9. **Rank 9:** Review for low search visibility. It has 102 February impressions and is in the lowest-volume group. It could be wrong if its low volume is expected rather than a sign of content risk.
10. **Rank 10:** Review for low search visibility. It has 102 February impressions and is prioritized because lower-volume pages showed higher observed `went_dark` rates. It could be wrong if the low visibility is unrelated to the content itself.
11. **Rank 11:** Review for low search visibility. It remains near the lowest-volume end of the queue and is prioritized by the confirmed volume signal. It could be wrong if low impressions reflect limited search demand.
12. **Rank 12:** Review for low search visibility. Its priority comes from the observed relationship between lower February impressions and higher March `went_dark` rates. It could be wrong if the low visibility is expected for the topic.
13. **Rank 13:** Review for low search visibility. It receives a high baseline score because of its relatively low February impression volume. It could be wrong if there is no actionable content issue behind the low visibility.
14. **Rank 14:** Review for low search visibility. It is prioritized using only February search visibility information. It could be wrong if the page's low impression volume is caused by limited demand.
15. **Rank 15:** Review for low search visibility. It remains among the highest-priority pages under the inverse-volume baseline. It could be wrong if the page is appropriately performing for its niche.
16. **Rank 16:** Review for low search visibility. Its ranking reflects the confirmed directional relationship between lower February impressions and higher observed `went_dark` rates. It could be wrong if measurement or topic demand explains the low volume.
17. **Rank 17:** Review for low search visibility. It receives priority because its February search visibility is relatively low. It could be wrong if the low visibility does not indicate a content refresh opportunity.
18. **Rank 18:** Review for low search visibility. The baseline prioritizes it based on February impression volume only. It could be wrong if the page naturally has limited search demand.
19. **Rank 19:** Review for low search visibility. It is still within the top 20 because lower-volume pages receive higher baseline scores. It could be wrong if low impressions are expected for the page's topic.
20. **Rank 20:** Review for low search visibility. It is included in the top 20 because of its relatively low February search visibility. It could be wrong if the low volume is unrelated to content quality or refresh need.

These reviews are decision-support recommendations, not proof that any individual page needs a refresh. A human review should consider topic demand, measurement coverage, and the actual content before taking action.


In [12]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Review the top 20 ranked items.
# March went_dark is shown only for audit/review, not for scoring.

top20 = baseline.head(20).copy()

review_cols = [
    "rank",
    "client_hash_id",
    "content_hash_id",
    "score",
    "gsc_impressions_feb",
    "gsc_clicks_feb",
    "avg_position_feb",
    "ctr_feb",
    "ctr_gap",
    "went_dark"
]

top20[review_cols]

,rank,client_hash_id,content_hash_id,score,gsc_impressions_feb,gsc_clicks_feb,avg_position_feb,ctr_feb,ctr_gap,went_dark
0,1,client_65de48885f4ef01b,content_3d3f76287d8a0cb4,0.216679,100.0,3.0,8.080000,0.030000,0.026203,1
1,2,client_23a62021009f63c4,content_666722e0d2f4dec0,0.216679,100.0,3.0,3.840000,0.030000,0.026203,0
2,3,client_23a62021009f63c4,content_df0749df761a54e6,0.216679,100.0,4.0,24.420000,0.040000,0.037912,0
3,4,client_3ffa76342f366962,content_071584d02651c65e,0.216217,101.0,7.0,5.980198,0.069307,0.065509,0
4,5,client_20259bd6705d81d4,content_ce070943bbb0ffa5,0.216217,101.0,4.0,5.871287,0.039604,0.035806,0
5,6,client_3ffa76342f366962,content_8cf83f905a2c7f95,0.215762,102.0,4.0,6.872549,0.039216,0.035418,1
6,7,client_3ffa76342f366962,content_7b17975c58745266,0.215762,102.0,5.0,4.450980,0.049020,0.045222,0
7,8,client_2094c6eb080311d5,content_696689d65af472ce,0.215762,102.0,3.0,4.666667,0.029412,0.025614,0
8,9,client_2094c6eb080311d5,content_e4163214c04a65d8,0.215762,102.0,6.0,2.196078,0.058824,0.054995,0
9,10,client_a80fca3f171ed1de,content_47b9e3d74d302801,0.215762,102.0,3.0,13.852941,0.029412,0.025326,0


## 4. Weak picks + leakage check

The weakest picks are the five lowest-ranked pages in the baseline queue. They have the highest February impression volumes and therefore receive the lowest priority under this baseline's inverse-volume scoring rule.

A weak pick would occur if a page has low review priority under the volume rule but would actually benefit from a content review. High February visibility does not guarantee that a page is healthy or that it does not need refreshing.

### Leakage check

The baseline score uses only `gsc_impressions_feb`, which belongs to the February feature window. The March `went_dark` outcome is retained only for evaluation and is not used to construct the score.

The leakage check confirms:

* The score uses only `gsc_impressions_feb`.
* March `went_dark` is used only for evaluation.
* The score does not contain the March outcome.
* The calculated score matches the February-only formula.

Therefore, the baseline is reproducible from February information alone and does not leak the March outcome into the scoring process.


In [13]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Weak-pick review
# These are the lowest-ranked items that still receive the baseline action.

weak_picks = baseline.tail(5).copy()

print("WEAK PICKS — LOWEST PRIORITY ITEMS")
print(
    weak_picks[
        [
            "rank",
            "client_hash_id",
            "content_hash_id",
            "score",
            "gsc_impressions_feb",
            "went_dark"
        ]
    ].to_string(index=False)
)

# Leakage check:
# March went_dark must not be part of the score calculation.
score_columns = [
    "gsc_impressions_feb"
]

print("\nLEAKAGE CHECK")
print("Score uses only:", score_columns)
print("March went_dark is used only for evaluation:", "went_dark" in baseline.columns)
print(
    "Score contains March outcome:",
    baseline["score"].equals(baseline["went_dark"])
)

# Verify the score is reproducible from February impressions alone.
recomputed_score = 1 / np.log1p(baseline["gsc_impressions_feb"])

print(
    "Score matches February-only formula:",
    np.allclose(baseline["score"], recomputed_score)
)

WEAK PICKS — LOWEST PRIORITY ITEMS
 rank          client_hash_id          content_hash_id    score  gsc_impressions_feb  went_dark
29349 client_62f4a7e64f5e0096 content_f107e54b10b43725 0.083621             156163.0          0
29350 client_62f4a7e64f5e0096 content_b99ea6861864dea5 0.083422             160699.0          0
29351 client_23a62021009f63c4 content_e8a52cf3d5988c07 0.083360             162129.0          0
29352 client_73cda7b4e4f265ea content_e241d6415ac9e534 0.083274             164152.0          0
29353 client_73cda7b4e4f265ea content_512dbad65bd5ade9 0.083142             167303.0          0

LEAKAGE CHECK
Score uses only: ['gsc_impressions_feb']
March went_dark is used only for evaluation: True
Score contains March outcome: False
Score matches February-only formula: True


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.